# UVEK 2026: review and reproduce the Premise workflow

**Review snapshot: 10 November 2026. Development goal paused.**

This notebook reviews the verified **UVEK 2026 cut-off + ecoinvent 3.10 biosphere** workflow for **REMIND SSP2-NPi, 2050**. It defaults to reading the existing three-sector export. It also contains the normal `NewDatabase` build and export steps, and a clearly separate internal emissions preview.

The verified mapping baseline is Premise commit **2528051f**. Later uncommitted generator/CHP mappings are **not qualified**: their broader residual check failed. Use the frozen verified package below rather than the dirty checkout. The full 19-sector goal remains incomplete.

**Prerequisites:** Python 3.12+, the reviewed Premise/Brightpath environment, the task-owned Brightway project with both background databases, a valid encrypted IAM file and key for fresh builds, and the CLIC cache/archive. Select kernel **Premise UVEK review (Python 3.12)**. No licensed inventories or credentials are embedded here.

Steps: configure → inspect inputs → optionally build/certify → optionally export → check links and signed inventories → optionally preview emissions → save a small review receipt.

In [ ]:
# Set these before importing Brightway or Premise. Restart the kernel if needed.
from pathlib import Path
import os
import sys
import json
import hashlib
import zipfile
import datetime
import copy
import importlib.metadata

REPO = Path.home() / "GitHub/premise"
TASK = Path("/private/tmp/premise-uvek-2026")
VERIFIED_PREMISE_ROOT = TASK / "emissions-gas-supply-final-wheel-install"
VERIFIED_WHEEL = (
    TASK / "emissions-gas-supply-final-wheels/premise-2.6.0-py3-none-any.whl"
)
EXPECTED_WHEEL_SHA256 = (
    "a01a44cffb7361f4394517c9fed6fb955e3cb5298c4cda085e5f31b2cce289d2"
)
BW_ROOT = Path.home() / "Library/Application Support/premise/uvek-2026-brightway"
PROJECT = "premise-uvek-2026-biosphere310"
SOURCE_DB = "UVEK:2026-biosphere310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
EXISTING_EXPORT = "premise-uvek-2026-remind-SSP2-NPi-2050-heat-public-final-auto-wheel"
IAM_DIRECTORY = TASK / "heat-public-final-wheel-install/premise/data/iam_output_files"
IAM_FILE = IAM_DIRECTORY / "remind_SSP2-NPi.csv"
EXPECTED_IAM_SHA256 = "1720125d7890fc4dadda88cd3abc978b2421be764756b0a9624d6dac272f6aa4"
CLIC_CACHE = TASK / "clic-cache"
GAINS_PREVIEW_FILE = TASK / "uvek_emissions_real_gains.private.nc"
EXPECTED_GAINS_SHA256 = (
    "2a572490e0c17c92ae23c998c82fdf34cd107f30546c0fc045da44a9bc578384"
)
OUTPUT_DIR = TASK / "review-notebook-output"

RUN_BUILD = False
WRITE_BRIGHTWAY = False
RUN_EMISSIONS_PREVIEW = False
CLIC_OFFLINE = True
SECTORS = ["biomass", "electricity", "heat"]
EXPORT_NAME = "premise-uvek-review-" + datetime.datetime.now(
    datetime.timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer."
assert not any(
    x in sys.modules for x in ("bw2data", "premise")
), "Restart the kernel before running configuration."
assert BW_ROOT.is_dir(), "Configure an existing task-owned Brightway data directory."
assert (
    VERIFIED_WHEEL.is_file() and VERIFIED_PREMISE_ROOT.is_dir()
), "Configure the frozen verified wheel/install paths."
assert hashlib.sha256(VERIFIED_WHEEL.read_bytes()).hexdigest() == EXPECTED_WHEEL_SHA256
with zipfile.ZipFile(VERIFIED_WHEEL) as archive:
    package_files = [
        n
        for n in archive.namelist()
        if n.startswith("premise/") and not n.endswith("/")
    ]
    assert len(package_files) == 413
    for name in package_files:
        assert archive.read(name) == (VERIFIED_PREMISE_ROOT / name).read_bytes(), name
os.environ["BRIGHTWAY2_DIR"] = str(BW_ROOT)
sys.path.insert(0, str(VERIFIED_PREMISE_ROOT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    "Verified frozen package; build/export/preview flags:",
    RUN_BUILD,
    WRITE_BRIGHTWAY,
    RUN_EMISSIONS_PREVIEW,
)


## 1. Inspect the existing project and evidence

The project must already exist; this notebook never creates or selects a reference-import project. The source technosphere and biosphere are read-only. Default execution inspects the previously verified export, without rebuilding it.

Local `/private/tmp` defaults describe the reviewed machine. Adjust paths if the preserved task artifacts have moved. The checked wheel isolates the notebook from unfinished checkout mappings.

In [ ]:
import bw2data as bd
import numpy as np
from premise import NewDatabase
import premise

assert Path(premise.__file__).is_relative_to(VERIFIED_PREMISE_ROOT)
assert (
    PROJECT in bd.projects
), "Project missing: install the reviewed backgrounds in an owned project first."
bd.projects.set_current(PROJECT)
for name in (SOURCE_DB, BIOSPHERE):
    assert name in bd.databases, f"Missing required database: {name}"
source_counts_before = {name: len(bd.Database(name)) for name in (SOURCE_DB, BIOSPHERE)}
assert (
    source_counts_before[SOURCE_DB] == 11947 and source_counts_before[BIOSPHERE] == 4362
)
print(
    {
        "project": PROJECT,
        "source_activities": source_counts_before[SOURCE_DB],
        "biosphere_flows": source_counts_before[BIOSPHERE],
        "Premise": importlib.metadata.version("premise"),
    }
)

receipts = {
    "electricity": REPO
    / "docs/development/uvek-electricity-milestone-verification.json",
    "public_three_sector": REPO / "docs/development/uvek-heat-public-verification.json",
    "internal_emissions": REPO
    / "docs/development/uvek-emissions-gas-supply-verification.json",
}
for label, filename in receipts.items():
    receipt = json.loads(filename.read_text())
    print(label, receipt.get("status", receipt.get("milestone")))


## 2. Optional fresh build through the normal public API

Set `RUN_BUILD=True` to construct the reviewed scenario. This can take time and populate task caches. The encrypted input checksum is checked **before** asking for a key; there is no unencrypted fallback. The key is requested interactively and is neither printed nor saved.

The public native route currently supports biomass, electricity and heat for this reviewed scenario. Other sectors remain gated. No manual helper installation or private scope override is used. `CLIC_OFFLINE=True` reuses the preserved pinned cache; set it to `False` only if you intend to retrieve the required archive through the normal provider.

In [ ]:
ndb = None
if RUN_BUILD:
    assert (
        IAM_FILE.is_file()
    ), "Set IAM_DIRECTORY to the verified encrypted REMIND file."
    assert hashlib.sha256(IAM_FILE.read_bytes()).hexdigest() == EXPECTED_IAM_SHA256
    if CLIC_OFFLINE:
        assert (CLIC_CACHE / "archives").is_dir(), "Pinned CLIC archive cache missing."
    from getpass import getpass

    iam_key = getpass("Premise IAM decryption key (not saved): ").strip()
    if not iam_key:
        raise ValueError("An explicit decryption key is required.")
    try:
        ndb = NewDatabase(
            scenarios=[
                {
                    "model": "remind",
                    "pathway": "SSP2-NPi",
                    "year": 2050,
                    "filepath": str(IAM_DIRECTORY),
                }
            ],
            source_type="brightway",
            source_db=SOURCE_DB,
            source_family="uvek",
            source_version="2026",
            system_model="cutoff",
            biosphere_name=BIOSPHERE,
            key=iam_key,
            inventory_source="clic",
            clic_options={"cache_dir": CLIC_CACHE, "offline": CLIC_OFFLINE},
            keep_imports_uncertainty=True,
            keep_source_db_uncertainty=True,
            use_cached_database=True,
            use_cached_inventories=True,
            inventory_backend="compact",
            generate_reports=False,
            quiet=True,
        )
    finally:
        del iam_key
    ndb.update(SECTORS)
    validation = ndb.get_validation_report(0, exhaustive=True)
    assert validation.valid, "Review the validation report before exporting."
    print("Fresh public build certified:", SECTORS)
else:
    print("Fresh build skipped; using the existing verified export for review.")


## 3. Optional export, then choose the graph to inspect

Set `WRITE_BRIGHTWAY=True` only with `RUN_BUILD=True`. A new, uniquely named database is written through the public exporter; overwriting any existing database is refused. Otherwise a fresh build is checked in memory, or the existing export is loaded in read-only review mode.

In [ ]:
from premise.inventory.inventory_store import get_scenario_inventory

if WRITE_BRIGHTWAY:
    assert ndb is not None, "Enable RUN_BUILD first."
    assert EXPORT_NAME not in (SOURCE_DB, BIOSPHERE) and EXPORT_NAME not in bd.databases
    ndb.write_db_to_brightway(EXPORT_NAME)
    target_database = EXPORT_NAME
    graph = list(bd.Database(target_database).load().values())
    mode = "fresh_public_build_and_export_readback"
elif ndb is not None:
    target_database = None
    graph = list(get_scenario_inventory(ndb.scenarios[0]))
    mode = "fresh_public_build_in_memory"
else:
    assert (
        EXISTING_EXPORT in bd.databases
    ), "Configure EXISTING_EXPORT or enable a fresh build/export."
    target_database = EXISTING_EXPORT
    graph = list(bd.Database(target_database).load().values())
    mode = "existing_verified_export_readback_only"
assert graph, "The selected graph is empty."
print({"mode": mode, "activities": len(graph), "database": target_database})


## 4. Check complete links and signed inventories

The following independent sparse-matrix check preserves production reference quantities and all signed exchanges. It verifies each technosphere identity/link and the canonical biosphere, then solves positive and negative demands for two heat markets and cobalt sulfate. It reports LCI finiteness and residuals, **not LCIA scores or impact completeness**. Finite matrices do not establish scientific equivalence of proxies.

The historical electricity selection has nine roots/55 source pins; the expanded public three-sector graph has 12 roots/76 records. These are different scopes.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu


def activity_key(record):
    return record["database"], record["code"]


def identity(record):
    return (
        record.get("name"),
        record.get("reference product") or record.get("product"),
        record.get("location"),
        record.get("unit"),
    )


providers = {activity_key(a): a for a in graph}
assert len(providers) == len(graph), "Duplicate activity keys."
indices = {k: i for i, k in enumerate(providers)}
known_biosphere = set(bd.Database(BIOSPHERE).load())
flow_keys = sorted(
    {
        tuple(e["input"])
        for a in graph
        for e in a["exchanges"]
        if e["type"] == "biosphere"
    }
)
flow_index = {k: i for i, k in enumerate(flow_keys)}
ar, ac, av, br, bc, bv = [], [], [], [], [], []
negative_inputs = 0
for column, activity in enumerate(graph):
    outputs = [e for e in activity["exchanges"] if e["type"] == "production"]
    assert len(outputs) == 1 and identity(outputs[0]) == identity(activity)
    assert outputs[0]["amount"] != 0
    for exchange in activity["exchanges"]:
        supplier, amount = tuple(exchange["input"]), exchange["amount"]
        assert not isinstance(amount, bool) and np.isfinite(amount)
        if exchange["type"] in ("production", "technosphere"):
            assert supplier in providers and identity(exchange) == identity(
                providers[supplier]
            )
            ar.append(indices[supplier])
            ac.append(column)
            av.append(amount if exchange["type"] == "production" else -amount)
            negative_inputs += int(exchange["type"] == "technosphere" and amount < 0)
        else:
            assert exchange["type"] == "biosphere" and supplier in known_biosphere
            br.append(flow_index[supplier])
            bc.append(column)
            bv.append(amount)
A = coo_matrix((av, (ar, ac)), shape=(len(graph), len(graph))).tocsc()
B = coo_matrix((bv, (br, bc)), shape=(len(flow_keys), len(graph))).tocsc()
solver = splu(A)
DEMANDS = [
    (
        "market for heat, for buildings",
        "heat, central or small-scale",
        "World",
        "megajoule",
    ),
    (
        "market for heat, district or industrial",
        "heat, district or industrial",
        "World",
        "megajoule",
    ),
    ("Cobalt sulfate", "Cobalt sulfate", "GLO", "kilogram"),
]
checks = []
for wanted in DEMANDS:
    matches = [a for a in graph if identity(a) == wanted]
    assert len(matches) == 1, f"Missing or ambiguous demand: {wanted}"
    inventories = []
    for quantity in (1.0, -1.0):
        demand = np.zeros(len(graph))
        demand[indices[activity_key(matches[0])]] = quantity
        supply = solver.solve(demand)
        inventory = B @ supply
        residual = float(np.max(np.abs(A @ supply - demand)))
        assert (
            np.isfinite(supply).all()
            and np.isfinite(inventory).all()
            and np.any(inventory != 0)
        )
        assert residual < 1e-7, (wanted, quantity, residual)
        checks.append(
            {
                "identity": wanted,
                "demand": quantity,
                "finite_LCI": True,
                "maximum_absolute_residual": residual,
            }
        )
        inventories.append(inventory)
    assert np.allclose(inventories[0], -inventories[1], rtol=1e-12, atol=0)
print(
    {
        "complete_links": True,
        "activities": len(graph),
        "signed_finite_LCIs": len(checks),
        "negative_technosphere_inputs_retained": negative_inputs,
    }
)


## 5. Optional internal emissions preview — not public sector support

This is a review tool for the **committed** internal callback: 3,487 reviewed inputs, 289 owned variants, 483 gaseous-air changes and 196 signed checks in its separate qualification. It does not enable the public emissions gate, complete missing mappings, or certify a four-sector export.

`RUN_EMISSIONS_PREVIEW=True` uses a separate copy of the graph. Without a fresh build it needs the preserved, hash-checked GAINS array derived from the verified encrypted scenario. The draft 137-input generator/CHP batch is excluded by the frozen package. No preview database is written.

In [ ]:
preview_summary = None
if RUN_EMISSIONS_PREVIEW:
    import warnings
    import xarray as xr
    from types import SimpleNamespace
    from premise.emissions.emissions import _update_emissions
    from premise.core.utils import scenario_metadata
    from premise.validation.sector_validation import validate_sector_contract

    if ndb is not None:
        iam = ndb.scenarios[0]["iam data"]
    else:
        assert GAINS_PREVIEW_FILE.is_file()
        assert (
            hashlib.sha256(GAINS_PREVIEW_FILE.read_bytes()).hexdigest()
            == EXPECTED_GAINS_SHA256
        )
        iam = SimpleNamespace(gains_data_IAM=xr.load_dataarray(GAINS_PREVIEW_FILE))
    preview = {
        "model": "remind",
        "pathway": "SSP2-NPi",
        "year": 2050,
        "database": copy.deepcopy(graph),
        "iam data": iam,
    }
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter("always")
        _update_emissions(preview, "2026", "cutoff", "CLE", "uvek", "3.10")
    assert validate_sector_contract(preview, "emissions").valid
    preview_summary = dict(preview["emissions diagnostics"])
    metadata = scenario_metadata(preview, "2026", "cutoff")
    assert metadata["scientific_status"] == "screening_only"
    assert (
        metadata["native_sector_audits"]["emissions"]["full_sector_mapping_complete"]
        is False
    )
    print(
        {
            "reviewed_inputs": preview_summary["caller_inputs"],
            "gaseous_air_changes": preview_summary["gaseous_air_changes"],
            "warnings": len(observed),
            "public_emissions_support": False,
        }
    )
else:
    print("Internal emissions preview skipped. Public workflow remains three sectors.")


## 6. Save the review receipt and inspect limitations

This small local JSON receipt contains identifiers, checks and counts, not inventory coefficients or a decryption key. Activity counts before/after are a quick check; they are not a full scientific-content hash. Historical source immutability evidence is in the linked verification reports.

**Review exercise:** compare a negative input's sign with its supplier's production reference before interpreting a negative score. Do not take absolute values: waste-treatment sign conventions, allocation and credits must be considered together. The answer is case-specific; this notebook does not infer an LCIA score from an input sign.

Remaining review topics: explicit proxy acceptability; final omissions/partial paperboard mappings and the separate CSP nitrate credit; upstream biosphere exclusions; 11 heat-role gaps; particles/VOC/direct emissions; the paused batch's residual failure; and the other 15 unqualified sectors. Cross-system-model conversions remain excluded.

In [ ]:
source_counts_after = {name: len(bd.Database(name)) for name in (SOURCE_DB, BIOSPHERE)}
assert source_counts_after == source_counts_before
review_receipt = {
    "review_date": "2026-11-10",
    "verified_mapping_commit": "2528051f",
    "execution_mode": mode,
    "source_family": "uvek",
    "source_release": "2026",
    "system_model": "cutoff",
    "biosphere": BIOSPHERE,
    "model": "remind",
    "pathway": "SSP2-NPi",
    "year": 2050,
    "sectors": SECTORS,
    "activities": len(graph),
    "complete_links": True,
    "signed_LCI_checks": checks,
    "source_activity_counts_unchanged": True,
    "fresh_public_build_executed": RUN_BUILD,
    "new_Brightway_export_written": WRITE_BRIGHTWAY,
    "internal_emissions_preview_executed": RUN_EMISSIONS_PREVIEW,
    "scientific_status": "screening_only",
    "all19sectors_operational": False,
}
receipt_path = OUTPUT_DIR / "uvek-review-notebook-run.json"
receipt_path.write_text(json.dumps(review_receipt, indent=2) + "\n")
print("Review receipt:", receipt_path)
print("Goal remains paused; no background source database was written.")
